In [0]:
import pandas as pd

file_path = "/Volumes/bootcamp_students/dataexpert_student_viknim/bronze_landing/fips_reference/canonical_national_county2020.txt"

pdf = pd.read_csv(file_path, sep="|", header=0, dtype=str)
df = spark.createDataFrame(pdf)

display(df)

In [0]:
from pyspark.sql import functions as F

cleaned_df = (
    df
    .withColumn("state_clean", F.upper(F.trim(F.col("STATE"))))
    .withColumn("county_name_upper", F.upper(F.trim(F.col("COUNTYNAME"))))
    
    # Extract the ending jurisdiction type before removing it
    .withColumn(
        "geo_type",
        F.when(F.col("county_name_upper").rlike(r"\s+COUNTY$"), "COUNTY")
         .when(F.col("county_name_upper").rlike(r"\s+CITY$"), "CITY")
         .when(F.col("county_name_upper").rlike(r"\s+MUNICIPIO$"), "MUNICIPIO")
         .when(F.col("county_name_upper").rlike(r"\s+PARISH$"), "PARISH")
         .when(F.col("county_name_upper").rlike(r"\s+CENSUS AREA$"), "CENSUS AREA")
         .when(F.col("county_name_upper").rlike(r"\s+BOROUGH$"), "BOROUGH")
         .otherwise("OTHER")
    )
    
    # Remove every recognized geographic suffix, not just County
    .withColumn(
        "place_name",
        F.trim(
            F.regexp_replace(
                F.col("county_name_upper"),
                r"\s+(COUNTY|CITY|MUNICIPIO|PARISH|CENSUS AREA|BOROUGH)$",
                ""
            )
        )
    )
    
    # Standardize remaining punctuation/spacing for matching
    .withColumn(
        "place_key",
        F.regexp_replace(F.col("place_name"), r"[^A-Z0-9]", "")
    )
)

display(
    cleaned_df.select(
        "STATE", "COUNTYNAME", "geo_type", "place_name", "place_key"
    )
)

In [0]:
same_name_regions = (
    cleaned_df
    .groupBy("state_clean", "place_key")
    .agg(
        F.count("*").alias("region_count"),
        F.collect_set("COUNTYNAME").alias("region_names"),
        F.collect_set("geo_type").alias("region_types"),
        F.collect_set("COUNTYFP").alias("county_fips_codes")
    )
    .filter(F.col("region_count") > 1)
    .orderBy("state_clean", "place_key")
)

display(same_name_regions)

In [0]:
same_name_keys = (
    same_name_regions
    .select("state_clean", "place_key")
)

same_name_rows = (
    cleaned_df
    .join(same_name_keys, ["state_clean", "place_key"], "inner")
    .select(
        "STATE",
        "STATEFP",
        "COUNTYFP",
        "COUNTYNS",
        "COUNTYNAME",
        "CLASSFP",
        "FUNCSTAT",
        "geo_type",
        "place_name",
        "place_key"
    )
    .orderBy("STATE", "place_key", "geo_type", "COUNTYNAME")
)

display(same_name_rows)